# Étape 3 — Première LoRA

**Projet 5 · LoRA Zellige** — Google Colab, GPU T4.

Objectif volontairement modeste : obtenir **une** LoRA qui fonctionne, de
bout en bout. On ne cherche pas encore le meilleur réglage — les
comparaisons entre rangs et nombres de pas sont l'affaire de l'Étape 4.

Prérequis : l'Étape 2 doit avoir produit la ligne `baseline` dans
`results/results.csv`, sinon il n'y a rien à comparer.

> **Exécution → Modifier le type d'exécution → GPU T4**


## 1. Préparer l'environnement


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors bitsandbytes lpips pyyaml


In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git pull -q)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py


## 2. Entraîner

Ce qui se passe à chaque pas :

1. on prend une image, le **VAE** la compresse en latent (512×512×3 → 64×64×4)
2. on y ajoute du bruit, d'une intensité tirée au hasard
3. le **U-Net** essaie de deviner le bruit ajouté, en lisant la légende
4. on compare sa réponse au vrai bruit, et on corrige les matrices LoRA

Tout le modèle est **gelé**. Seules les matrices LoRA apprennent : environ
**1,5 million de paramètres sur 860 millions**, soit 0,2 %.

Configuration : rang 8, 1000 pas, lot effectif de 4. Comptez 15 à 25 min.
Un point de contrôle est écrit tous les 250 pas, au cas où Colab coupe.


In [ ]:
!python src/train_lora.py --config configs/lora_r8.yaml


## 3. La courbe de perte

Elle doit **descendre puis se stabiliser**. Si elle reste plate, le taux
d'apprentissage est trop faible. Si elle oscille violemment, il est trop fort.

Attention : une perte qui descend ne prouve pas que le style est appris.
Elle dit seulement que le modèle prédit mieux le bruit. Le vrai juge, ce
sont les mesures de l'étape suivante.


In [ ]:
import json, matplotlib.pyplot as plt

rapport = json.load(open('runs/r8/entrainement.json'))
courbe = rapport['courbe']

plt.figure(figsize=(9, 3.5))
plt.plot([c['pas'] for c in courbe], [c['perte'] for c in courbe])
plt.xlabel('pas'); plt.ylabel('perte (MSE sur le bruit)')
plt.title(f"{rapport['nom']} — perte finale {rapport['perte_finale']}")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

for cle in ['parametres_entrainables', 'part_entrainable_pct',
            'duree_s', 'duree_par_pas_s', 'vram_pic_go', 'taille_lora_mo']:
    print(f'{cle:<26} {rapport[cle]}')


## 4. Générer avec la LoRA

Mêmes 10 prompts, mêmes 4 graines que la baseline. Rien d'autre ne change,
donc tout écart observé vient de la LoRA et d'elle seule.


In [ ]:
!python src/generate.py --lora runs/r8 --intensite 1.0 --sortie results/r8


## 5. Mesurer


In [ ]:
!python src/evaluate.py --dossier results/r8 --nom lora_r8_1000pas


## 6. Avant / après

La comparaison qui compte. Même prompt, même graine, seule la LoRA diffère.


In [ ]:
from PIL import Image
import yaml, matplotlib.pyplot as plt

prompts = yaml.safe_load(open('configs/eval_prompts.yaml'))['prompts']
graine = 1234

fig, axes = plt.subplots(2, len(prompts), figsize=(2.6 * len(prompts), 6))
for colonne, p in enumerate(prompts):
    for ligne, dossier in enumerate(['results/baseline', 'results/r8']):
        ax = axes[ligne, colonne]
        ax.imshow(Image.open(f"{dossier}/{p['id']}_s{graine}.png"))
        ax.axis('off')
        if ligne == 0:
            ax.set_title(f"{p['id']}\n{p['categorie']}", fontsize=9)
axes[0, 0].set_ylabel('avant'); axes[1, 0].set_ylabel('apres')
fig.text(0.005, 0.74, 'AVANT', rotation=90, fontsize=12, weight='bold')
fig.text(0.005, 0.26, 'APRES', rotation=90, fontsize=12, weight='bold')
plt.suptitle('Baseline SD 1.5 vs LoRA rang 8 — graine 1234', fontsize=13)
plt.tight_layout(); plt.show()


## 7. Les chiffres, côte à côte


In [ ]:
import pandas as pd

df = pd.read_csv('results/results.csv').set_index('nom')
colonnes = ['style_dino_moy', 'style_clip_moy', 'prompt_domaine',
            'prompt_transfert', 'prompt_hors_domaine',
            'memorisation_max_dino', 'diversite_lpips']
tableau = df[colonnes].T
if 'lora_r8_1000pas' in tableau.columns:
    tableau['ecart'] = (tableau['lora_r8_1000pas'] - tableau['baseline']).round(4)
tableau


### Comment lire cet écart

| Ligne | Ce qu'on veut voir | Pourquoi |
|---|---|---|
| `style_dino_moy` | **nette hausse** | c'est la preuve que le style est appris |
| `prompt_domaine` | baisse modérée tolérée | la LoRA pousse son style au détriment du texte |
| `prompt_transfert` | **ne doit pas s'effondrer** | sinon le modèle recopie des panneaux au lieu d'avoir appris un style |
| `prompt_hors_domaine` | **stable** | une chute = la LoRA a abîmé le modèle de base |
| `memorisation_max_dino` | proche de 0,737 | au-delà, suspicion de recopie d'une image d'entraînement |
| `diversite_lpips` | **ne doit pas s'effondrer** | sinon les 4 graines donnent la même image |

Si `style_dino_moy` monte et que tout le reste tient : l'Étape 3 est réussie.


## 8. Sauvegarder

Le fichier LoRA et les 40 images restent sur Colab (trop lourds pour Git).
On ne renvoie que les mesures.


In [ ]:
from google.colab import userdata
jeton = userdata.get('GITHUB_TOKEN')

!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/results.csv results/r8/manifeste.json results/r8/evaluation.json \
         runs/r8/entrainement.json
!git commit -m 'etape3: premiere LoRA rang 8, 1000 pas'
!git push https://{jeton}@github.com/HatimTj/lora-zellige-patrimoine.git main


> Pensez à **effacer la sortie** de la cellule ci-dessus avant d'enregistrer
> le notebook : en cas d'échec, git affiche l'URL du dépôt, jeton compris.


## 9. Télécharger la LoRA

Gardez le fichier : il sert à la démo de l'Étape 6, et le réentraîner coûte
20 minutes de GPU.


In [ ]:
from google.colab import files
files.download('runs/r8/pytorch_lora_weights.safetensors')
